<a href="https://colab.research.google.com/github/lebogangreginald/MSc_AI/blob/main/CNN_ViT_Today2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install -y openslide-tools
!pip install openslide-python


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libopenslide0
Suggested packages:
  libtiff-tools
The following NEW packages will be installed:
  libopenslide0 openslide-tools
0 upgraded, 2 newly installed, 0 to remove and 1 not upgraded.
Need to get 104 kB of archives.
After this operation, 297 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenslide0 amd64 3.4.1+dfsg-5build1 [89.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openslide-tools amd64 3.4.1+dfsg-5build1 [13.8 kB]
Fetched 104 kB in 1s (125 kB/s)
Selecting previously unselected package libopenslide0.
(Reading database ... 117528 files and directories currently installed.)
Preparing to unpack .../libopenslide0_3.4.1+dfsg-5build1_amd64.deb ...
Unpacking libopenslide0 (3.4.1+dfsg-5build1) ...
Selecting previously unselected package openslide-tools.


In [ ]:
# Libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import openslide
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
# Dataset Loading
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path
import pandas as pd
import tensorflow as tf

DATA_ROOT = Path("/content/drive/MyDrive/prostate-gleason-dataset-master")

# Load CSVs using correct column names
df_train = pd.read_csv(DATA_ROOT / "train.csv")
df_test = pd.read_csv(DATA_ROOT / "test.csv")

# Rename if needed (to keep consistency)
df_train = df_train.rename(columns={"class": "label"})
df_test = df_test.rename(columns={"class": "label"})

print(df_train.head())
print(df_train["label"].value_counts())


                                         image  height_biopsy  width_biopsy  \
0  002a4db09dad406c85505a00fb6f6144_11265.jpeg          32773         23904   
1   002a4db09dad406c85505a00fb6f6144_1382.jpeg          32773         23904   
2  002a4db09dad406c85505a00fb6f6144_11261.jpeg          32773         23904   
3   002a4db09dad406c85505a00fb6f6144_3354.jpeg          32773         23904   
4  002a4db09dad406c85505a00fb6f6144_11360.jpeg          32773         23904   

   label  
0      0  
1      0  
2      0  
3      0  
4      0  
label
0    34303
1    14058
2     9921
3     6759
Name: count, dtype: int64


In [ ]:
TRAIN_IMG_DIR = DATA_ROOT / "train"
TEST_IMG_DIR = DATA_ROOT / "test"

df_train["path"] = df_train["image"].apply(lambda x: str(TRAIN_IMG_DIR / x))
df_test["path"] = df_test["image"].apply(lambda x: str(TEST_IMG_DIR / x))

train_paths = df_train["path"].values
train_labels = df_train["label"].values

test_paths = df_test["path"].values
test_labels = df_test["label"].values


In [ ]:
import os
print("Train exists:", os.path.exists(train_paths[0]))
print("Test exists:", os.path.exists(test_paths[0]))


Train exists: True
Test exists: True


In [ ]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=0.30,
    random_state=42,
    stratify=train_labels
)


In [ ]:
# Save Val split
import pandas as pd

# Create a validation dataframe
df_val = pd.DataFrame({
    "image": [Path(p).name for p in val_paths],
    "label": val_labels
})

# Save inside dataset folder for consistency
df_val_path = DATA_ROOT / "val.csv"
df_val.to_csv(df_val_path, index=False)

print("Validation CSV saved to:", df_val_path)
print(df_val.head())


Validation CSV saved to: /content/drive/MyDrive/prostate-gleason-dataset-master/val.csv
                                         image  label
0  52bbaa0fbe4b7a3d193fc41eec5b0f46_28672.jpeg      0
1   2a0f517e43256b1e6580c389d95cc96b_2975.jpeg      0
2   19565bef71aac6c8e9fa6638480c8ecd_3339.jpeg      0
3   235cc1d05d01d1a3d2424514b27e68e2_7661.jpeg      0
4   eb88d9362516b7ff394e7c3b41d2931d_4157.jpeg      2


In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 32

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)  # JPEG not PNG
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    return image, label

def create_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_paths, train_labels, training=True)
val_ds   = create_dataset(val_paths, val_labels)
test_ds  = create_dataset(test_paths, test_labels)


In [ ]:
from tensorflow.keras import layers, Model

# Vision Transformer Model Definition
def create_vit_classifier(
    input_shape=(256, 256, 3),
    patch_size=8,
    projection_dim=192,
    transformer_layers=8,
    num_heads=4,
    mlp_dim=256,
    num_classes=4,
    dropout_rate=0.1,
):
    num_patches = (input_shape[0] // patch_size) ** 2

    # Inputs
    inputs = layers.Input(shape=input_shape)

    # Patching
    patches = layers.Reshape((num_patches, patch_size * patch_size * 3))(inputs)

    # Patch encoding
    encoded = layers.Dense(projection_dim)(patches)
    positions = tf.range(start=0, limit=num_patches, delta=1)
    encoded += layers.Embedding(num_patches, projection_dim)(positions)

    # Transformer blocks
    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization()(encoded)
        attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim)(x1, x1)
        x2 = layers.Add()([attn, encoded])
        x3 = layers.LayerNormalization()(x2)
        mlp = layers.Dense(mlp_dim, activation=tf.nn.gelu)(x3)
        mlp = layers.Dense(projection_dim)(mlp)
        encoded = layers.Add()([mlp, x2])

    # Classification Head
    x = layers.LayerNormalization()(encoded)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs, outputs)

# Create & compile the model
vit_model = create_vit_classifier()

vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

vit_model.summary()


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_2 (Reshape) │ (None, 1024, 192) │          0 │ input_layer_8[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 1024, 192) │     37,056 │ reshape_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_13 (Add)        │ (None, 1024, 192) │          0 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1024, 192) │        384 │ add_13[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1024, 192) │    592,320 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_14 (Add)        │ (None, 1024, 192) │          0 │ multi_head_atten… │
│                     │                   │            │ add_13[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1024, 192) │        384 │ add_14[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 1024, 256) │     49,408 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 1024, 192) │     49,344 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_15 (Add)        │ (None, 1024, 192) │          0 │ dense_15[0][0],   │
│                     │                   │            │ add_14[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1024, 192) │        384 │ add_15[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1024, 192) │    592,320 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_16 (Add)        │ (None, 1024, 192) │          0 │ multi_head_atten… │
│                     │                   │            │ add_15[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1024, 192) │        384 │ add_16[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 1024, 256) │     49,408 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 1024, 192) │     49,344 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_17 (Add)        │ (None, 1024, 192) │          0 │ dense_17[0][0],   │
│                     │                   │            │ add_16[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1024, 192) │        384 │ add_17[0][0]      │
│ (LayerNormalizatio… │                   │            │                 

 Total params: 5,572,932 (21.26 MB)

 Trainable params: 5,572,932 (21.26 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

# Extract labels from train_ds to compute weights
train_labels = []
for _, labels in train_ds:
    train_labels.extend(labels.numpy())
train_labels = np.array(train_labels)

# Compute class weights
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = dict(enumerate(class_weights_arr))

print("Class Weights:", class_weights)


Class Weights: {0: np.float64(0.4740129935032484), 1: np.float64(1.1567073170731708), 2: np.float64(1.6388768898488122), 3: np.float64(2.405833861762841)}


In [ ]:
print(train_ds)
print(val_ds)
print(test_ds)


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>
<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))>


In [ ]:
print([k for k in globals().keys() if "train" in k.lower()])


['train_test_split', 'df_train', 'TRAIN_IMG_DIR', 'train_paths', 'train_labels', 'train_df', 'train_ds']


In [ ]:
import tensorflow as tf
import pandas as pd
from pathlib import Path

# ===============================
# PATHS
# ===============================
DATA_ROOT = Path("/content/drive/MyDrive/prostate-gleason-dataset-master")

TRAIN_IMG_DIR = DATA_ROOT / "train"
TEST_IMG_DIR  = DATA_ROOT / "test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# ===============================
# LOAD CSV FILES
# ===============================
train_df = pd.read_csv(DATA_ROOT / "train.csv")
val_df   = pd.read_csv(DATA_ROOT / "val.csv")
test_df  = pd.read_csv(DATA_ROOT / "test.csv")

print("Train CSV columns:", train_df.columns.tolist())
print("Val CSV columns:", val_df.columns.tolist())
print("Test CSV columns:", test_df.columns.tolist())

# ===============================
# IMAGE LOADER
# ===============================
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# ===============================
# DATASET BUILDER
# ===============================
def build_dataset(df, img_dir, shuffle=False):
    image_paths = df["image"].apply(lambda x: str(img_dir / x)).values
    labels = df["label"].values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df))

    ds = ds.map(
        lambda x, y: load_image(x, y),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

# ===============================
# CREATE DATASETS
# ===============================
train_ds = build_dataset(train_df, TRAIN_IMG_DIR, shuffle=True)
val_ds   = build_dataset(val_df,   TRAIN_IMG_DIR, shuffle=False)
test_ds  = build_dataset(test_df,  TEST_IMG_DIR,  shuffle=False)

print("✅ train_ds, val_ds, and test_ds are ready")


Train CSV columns: ['image', 'height_biopsy', 'width_biopsy', 'class']
Val CSV columns: ['image', 'label']
Test CSV columns: ['image', 'height_biopsy', 'width_biopsy', 'class']


KeyError: 'label'

In [ ]:
import tensorflow as tf

# ===============================
# ViT Model Builder (Keras-safe)
# ===============================
def build_vit(
    input_shape=(224, 224, 3),
    num_classes=4,
    patch_size=16,
    embed_dim=256,
    num_heads=4,
    mlp_dim=512,
    num_layers=6
):
    inputs = tf.keras.Input(shape=input_shape)

    # Patch embedding
    x = tf.keras.layers.Conv2D(
        embed_dim,
        kernel_size=patch_size,
        strides=patch_size,
        padding="valid"
    )(inputs)

    num_patches = (input_shape[0] // patch_size) ** 2  # 196

    x = tf.keras.layers.Reshape((num_patches, embed_dim))(x)

    # Positional embedding (SAFE)
    pos_embed = tf.keras.layers.Embedding(
        input_dim=num_patches,
        output_dim=embed_dim
    )(tf.range(start=0, limit=num_patches, delta=1))

    x = x + pos_embed

    # Transformer blocks
    for _ in range(num_layers):
        # Multi-head self-attention
        x1 = tf.keras.layers.LayerNormalization()(x)
        attn = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )(x1, x1)
        x = tf.keras.layers.Add()([x, attn])

        # MLP block
        x2 = tf.keras.layers.LayerNormalization()(x)
        mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(mlp_dim, activation="gelu"),
            tf.keras.layers.Dense(embed_dim),
        ])(x2)
        x = tf.keras.layers.Add()([x, mlp])

    # Classification head
    x = tf.keras.layers.LayerNormalization()(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)

# ===============================
# Build Model
# ===============================
vit_model = build_vit()

# ===============================
# Compile
# ===============================
vit_model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-4,
        weight_decay=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ===============================
# Callbacks
# ===============================
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "vit_best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# ===============================
# Train
# ===============================
history = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[checkpoint, early_stop],
    class_weight=class_weights,
    verbose=1
)


NameError: name 'class_weights' is not defined

In [ ]:
import tensorflow as tf
import math


# Warm-up + Cosine Decay Schedule
class WarmUpCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, total_steps, warmup_steps):
        super().__init__()
        self.base_lr = base_lr
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        step = tf.cast(step, tf.float32)

        # Warm-up phase
        warmup_lr = self.base_lr * (step / self.warmup_steps)

        # Cosine decay phase
        progress = (step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
        cosine_lr = 0.5 * self.base_lr * (1 + tf.cos(math.pi * progress))

        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)

# ===============================
# Training Parameters
# ===============================
epochs = 50

total_steps = epochs * steps_per_epoch
warmup_steps = int(0.1 * total_steps)   # 10% warm-up
base_lr = 1e-4

# ===============================
# Optimizer with Warm-up
# ===============================
lr_schedule = WarmUpCosine(
    base_lr=base_lr,
    total_steps=total_steps,
    warmup_steps=warmup_steps
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=1e-4
)

# ===============================
# Compile ViT Model
# ===============================
vit_model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ===============================
# Callbacks
# ===============================
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "vit_best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

callbacks = [checkpoint, early_stop]

# ===============================
# Train Model
# ===============================
history = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)


NameError: name 'steps_per_epoch' is not defined

In [ ]:
# ViT Train
# Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    "vit_best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    verbose=1
)

callbacks = [checkpoint, reduce_lr, early_stop]

# Training line
history = vit_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,   # EarlyStopping will end early if no improvement
    callbacks=callbacks,
    class_weight=class_weights,  # Balanced training
    verbose=1
)


Epoch 1/50
1423/1423 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3882 - loss: 1.3170
Epoch 1: val_accuracy improved from -inf to 0.47374, saving model to vit_best_model.keras
1423/1423 ━━━━━━━━━━━━━━━━━━━━ 6743s 5s/step - accuracy: 0.3882 - loss: 1.3169 - val_accuracy: 0.4737 - val_loss: 1.0713 - learning_rate: 1.0000e-04
Epoch 2/50
1423/1423 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.5194 - loss: 1.1130
Epoch 2: val_accuracy improved from 0.47374 to 0.47814, saving model to vit_best_model.keras
1423/1423 ━━━━━━━━━━━━━━━━━━━━ 4203s 3s/step - accuracy: 0.5194 - loss: 1.1130 - val_accuracy: 0.4781 - val_loss: 1.1442 - learning_rate: 1.0000e-04
Epoch 3/50
1333/1423 ━━━━━━━━━━━━━━━━━━━━ 3:52 3s/step - accuracy: 0.5723 - loss: 1.0223

In [ ]:
# Test evaluation
test_loss, test_acc = vit_model.evaluate(test_ds, verbose=1)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")
print(f"📉 Test Loss: {test_loss:.4f}")


In [ ]:
# Classification Report and Confusion Matrix
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Collect predictions
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = vit_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Classification Report
print("\n🔍 Classification Report")
print(classification_report(y_true, y_pred, digits=4))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - ViT")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()


In [ ]:
# Training Curves
# Plot accuracy and loss curves
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(1, len(acc) + 1)

plt.figure(figsize=(14,5))
plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label="Train Accuracy")
plt.plot(epochs_range, val_acc, label="Val Accuracy")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label="Train Loss")
plt.plot(epochs_range, val_loss, label="Val Loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


In [ ]:
#AUC Plot
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import numpy as np

# Number of classes
num_classes = 4

# Binarize true labels for multi-class ROC
y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))

# Get softmax prediction probabilities
y_pred_proba = []
for images, _ in test_ds:
    preds = vit_model.predict(images, verbose=0)
    y_pred_proba.append(preds)
y_pred_proba = np.vstack(y_pred_proba)

# Compute ROC + AUC per class
fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Macro AUC (unweighted mean)
macro_auc = np.mean(list(roc_auc.values()))
print(f"\n🎯 Macro AUC: {macro_auc:.4f}")


In [ ]:
#Plots
plt.figure(figsize=(8, 6))

colors = ['blue', 'orange', 'green', 'red']
class_names = ['Class 0', 'Class 1', 'Class 2', 'Class 3']  # Update if needed

for i, color in zip(range(num_classes), colors):
    plt.plot(
        fpr[i], tpr[i],
        color=color,
        lw=2,
        label=f"{class_names[i]} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Vision Transformer (One-vs-Rest)")
plt.legend(loc="lower right")
plt.show()


In [ ]:
#CNN

In [ ]:
from pathlib import Path
import pandas as pd
import tensorflow as tf

DATA_ROOT = Path("/content/drive/MyDrive/prostate-gleason-dataset-master")

# Load CSVs using correct column names
df_train = pd.read_csv(DATA_ROOT / "train.csv")
df_test = pd.read_csv(DATA_ROOT / "test.csv")

# Rename if needed (to keep consistency)
df_train = df_train.rename(columns={"class": "label"})
df_test = df_test.rename(columns={"class": "label"})

print(df_train.head())
print(df_train["label"].value_counts())


                                         image  height_biopsy  width_biopsy  \
0  002a4db09dad406c85505a00fb6f6144_11265.jpeg          32773         23904   
1   002a4db09dad406c85505a00fb6f6144_1382.jpeg          32773         23904   
2  002a4db09dad406c85505a00fb6f6144_11261.jpeg          32773         23904   
3   002a4db09dad406c85505a00fb6f6144_3354.jpeg          32773         23904   
4  002a4db09dad406c85505a00fb6f6144_11360.jpeg          32773         23904   

   label  
0      0  
1      0  
2      0  
3      0  
4      0  
label
0    34303
1    14058
2     9921
3     6759
Name: count, dtype: int64


In [ ]:


import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

# Dataset paths
DATA_ROOT = Path("/content/drive/MyDrive/prostate-gleason-dataset-master")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

# Load CSVs
train_df = pd.read_csv(DATA_ROOT / "train.csv")
test_df = pd.read_csv(DATA_ROOT / "test.csv")

# Create file paths
train_df["filepath"] = train_df["image"].apply(lambda x: str(TRAIN_DIR / x))
test_df["filepath"] = test_df["image"].apply(lambda x: str(TEST_DIR / x))

print("Training samples:", len(train_df))
print("Test samples:", len(test_df))



Training samples: 65041
Test samples: 7416


In [ ]:
# Train/Val Split
# Split: 70% train / 20% val / 10% test internal
train_split, temp_split = train_test_split(
    train_df,
    test_size=0.30,
    stratify=train_df["class"],
    random_state=42
)

val_df, test_df_local = train_test_split(
    temp_split,
    test_size=0.33,
    stratify=temp_split["class"],
    random_state=42
)

# Convert labels to string for Keras
train_split["class"] = train_split["class"].astype(str)
val_df["class"] = val_df["class"].astype(str)

# Compute class weights for imbalance
class_weights = class_weight.compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_split["class"]),
    y=train_split["class"]
)
class_weights = dict(enumerate(class_weights))

print("Train:", len(train_split), "Val:", len(val_df), "Test:", len(test_df_local))
print("Class Weights:", class_weights)


Train: 45528 Val: 13073 Test: 6440
Class Weights: {0: np.float64(0.4740129935032484), 1: np.float64(1.1567073170731708), 2: np.float64(1.6388768898488122), 3: np.float64(2.405833861762841)}


In [ ]:
# Save the splits to CSV files
# Paths to save split files
(train_split_path,
 val_split_path,
 internal_test_split_path) = (
    "/content/drive/MyDrive/train_split.csv",
    "/content/drive/MyDrive/val_split.csv",
    "/content/drive/MyDrive/internal_test_split.csv"
)

# Save the splits
train_split.to_csv(train_split_path, index=False)
val_df.to_csv(val_split_path, index=False)
test_df_local.to_csv(internal_test_split_path, index=False)

print("Saved Splits:")
print("Train:", train_split_path)
print("Val:", val_split_path)
print("Test:", internal_test_split_path)


Saved Splits:
Train: /content/drive/MyDrive/train_split.csv
Val: /content/drive/MyDrive/val_split.csv
Test: /content/drive/MyDrive/internal_test_split.csv


In [ ]:
#Compute Class Weights for CNN Training ===
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Extract target labels from your train_split dataframe
train_labels_cnn = train_split["class"].values

# Compute balanced class weights
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels_cnn),
    y=train_labels_cnn
)

class_weights = dict(enumerate(class_weights_arr))

print("Class Weights for CNN:", class_weights)



Class Weights for CNN: {0: np.float64(0.4740129935032484), 1: np.float64(1.1567073170731708), 2: np.float64(1.6388768898488122), 3: np.float64(2.405833861762841)}


In [ ]:
# Train ResNet50
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4

# Data generators
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.2
).flow_from_dataframe(
    train_split,
    x_col="filepath",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
).flow_from_dataframe(
    val_df,
    x_col="filepath",
    y_col="class",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

# Build model
base_model = ResNet50(include_top=False, weights="imagenet", input_shape=(*IMG_SIZE, 3))
base_model.trainable = False  # freeze base

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.3)(x)
output = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model_resnet = models.Model(inputs=base_model.input, outputs=output)

model_resnet.compile(
    optimizer=optimizers.Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Save checkpoint to Drive ✔
ckpt = ModelCheckpoint(
    "/content/drive/MyDrive/best_resnet50.h5",
    save_best_only=True,
    monitor="val_accuracy",
    mode="max"
)

callbacks = [
    ckpt,
    EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=4)
]

# TRAIN
history_resnet = model_resnet.fit(
    train_gen,
    validation_data=val_gen,
    epochs=3,
    class_weight=class_weights,
    callbacks=callbacks
)

print("✔ Training Finished — model saved → /content/drive/MyDrive/best_resnet50.h5")


Found 45527 validated image filenames belonging to 4 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/legacy/preprocessing/image.py:920: UserWarning: Found 1 invalid image filename(s) in x_col="filepath". These filename(s) will be ignored.
  warnings.warn(


Found 13073 validated image filenames belonging to 4 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/3
1423/1423 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.4832 - loss: 1.2750

1423/1423 ━━━━━━━━━━━━━━━━━━━━ 18011s 13s/step - accuracy: 0.4832 - loss: 1.2748 - val_accuracy: 0.7355 - val_loss: 0.6405 - learning_rate: 1.0000e-04
Epoch 2/3
1423/1423 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.6966 - loss: 0.7979

1423/1423 ━━━━━━━━━━━━━━━━━━━━ 7801s 5s/step - accuracy: 0.6966 - loss: 0.7979 - val_accuracy: 0.7660 - val_loss: 0.5691 - learning_rate: 1.0000e-04
Epoch 3/3
1423/1423 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7278 - loss: 0.7282

1423/1423 ━━━━━━━━━━━━━━━━━━━━ 7985s 6s/step - accuracy: 0.7278 - loss: 0.7282 - val_accuracy: 0.7977 - val_loss: 0.5102 - learning_rate: 1.0000e-04
✔ Training Finished — model saved → /content/drive/MyDrive/best_resnet50.h5


In [ ]:
# Test evaluation
test_loss_resnet, test_acc_resnet = model_resnet.evaluate(test_gen, verbose=1)
print(f"\n🔥 ResNet50 Test Accuracy: {test_acc_resnet:.4f}")
print(f"📉 Test Loss: {test_loss_resnet:.4f}")


In [ ]:
import numpy as np

# Predictions
pred_probs = model_resnet.predict(test_gen, verbose=1)
pred_labels = np.argmax(pred_probs, axis=1)

# Ground truth
true_labels = test_gen.classes

class_indices = test_gen.class_indices
class_names = list(class_indices.keys())


In [ ]:
# Classification Report
from sklearn.metrics import classification_report

print("\n🔍 Classification Report — ResNet50 CNN")
print(classification_report(true_labels, pred_labels, target_names=class_names, digits=4))


In [ ]:
# Confusion Matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, pred_labels)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix — ResNet50 CNN")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


In [ ]:
# AUC
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

num_classes = len(class_names)
y_true_bin = label_binarize(true_labels, classes=np.arange(num_classes))

fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

macro_auc_resnet = np.mean(list(roc_auc.values()))
print(f"\n🎯 ResNet50 Macro AUC: {macro_auc_resnet:.4f}")


In [ ]:
plt.figure(figsize=(8,6))
colors = ['blue', 'orange', 'green', 'red']

for i, color in zip(range(num_classes), colors):
    plt.plot(fpr[i], tpr[i], lw=2, color=color,
             label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})')

plt.plot([0,1],[0,1],'k--')
plt.xlim([0,1])
plt.ylim([0,1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — ResNet50 CNN")
plt.legend(loc="lower right")
plt.show()


In [ ]:
# Training Curves
acc = history_resnet.history['accuracy']
val_acc = history_resnet.history['val_accuracy']
loss = history_resnet.history['loss']
val_loss = history_resnet.history['val_loss']

epochs = range(1, len(acc)+1)

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(epochs, acc, label="Train Accuracy")
plt.plot(epochs, val_acc, label="Val Accuracy")
plt.title("Accuracy Curve — ResNet50 CNN")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs, loss, label="Train Loss")
plt.plot(epochs, val_loss, label="Val Loss")
plt.title("Loss Curve — ResNet50 CNN")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()


In [ ]:
# Summary
print(f"""
📌 End-to-End CNN Results (ResNet50)
-----------------------------------
Test Accuracy: {test_acc_resnet:.4f}
Macro AUC:     {macro_auc_resnet:.4f}
""")


In [ ]:
#Hybrid CNN-ViT

In [ ]:
!apt-get install -y openslide-tools
!pip install openslide-python


In [ ]:
# Libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import openslide
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

In [ ]:
# Dataset Loading
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd
import tensorflow as tf

DATA_ROOT = Path("/content/drive/MyDrive/prostate-gleason-dataset-master")

# Load CSVs using correct column names
df_train = pd.read_csv(DATA_ROOT / "train.csv")
df_test = pd.read_csv(DATA_ROOT / "test.csv")

# Rename if needed (to keep consistency)
df_train = df_train.rename(columns={"class": "label"})
df_test = df_test.rename(columns={"class": "label"})

print(df_train.head())
print(df_train["label"].value_counts())

In [ ]:
TRAIN_IMG_DIR = DATA_ROOT / "train"
TEST_IMG_DIR = DATA_ROOT / "test"

df_train["path"] = df_train["image"].apply(lambda x: str(TRAIN_IMG_DIR / x))
df_test["path"] = df_test["image"].apply(lambda x: str(TEST_IMG_DIR / x))

train_paths = df_train["path"].values
train_labels = df_train["label"].values

test_paths = df_test["path"].values
test_labels = df_test["label"].values

In [ ]:
from sklearn.model_selection import train_test_split

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths, train_labels,
    test_size=0.30,
    random_state=42,
    stratify=train_labels
)

In [ ]:
# Save Val split
import pandas as pd

# Create a validation dataframe
df_val = pd.DataFrame({
    "image": [Path(p).name for p in val_paths],
    "label": val_labels
})

# Save inside dataset folder for consistency
df_val_path = DATA_ROOT / "val.csv"
df_val.to_csv(df_val_path, index=False)

print("Validation CSV saved to:", df_val_path)
print(df_val.head())

In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 32

def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)  # JPEG not PNG
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    return image, label

def create_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1000)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_dataset(train_paths, train_labels, training=True)
val_ds   = create_dataset(val_paths, val_labels)
test_ds  = create_dataset(test_paths, test_labels)

In [ ]:
# MOdel
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
import tensorflow as tf


In [ ]:
def create_hybrid_resnet_vit(
    input_shape=(256, 256, 3),
    patch_size=16,
    projection_dim=64,
    transformer_layers=8,
    num_heads=4,
    mlp_dim=128,
    num_classes=4,
    dropout_rate=0.1,
):
    inputs = layers.Input(shape=input_shape)


    # CNN Feature Extractor

    resnet = ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )
    resnet.trainable = False  # Freeze base layers

    x = resnet(inputs)
    feature_map_shape = tf.shape(x)


    # Convert to patches (Patchify CNN features)

    patch_h = x.shape[1]
    patch_w = x.shape[2]
    num_patches = patch_h * patch_w
    projection_input_dim = x.shape[-1]

    x = layers.Reshape((num_patches, projection_input_dim))(x)


    # ViT Encoder

    # Linear projection
    encoded = layers.Dense(projection_dim)(x)
    positions = tf.range(start=0, limit=num_patches, delta=1)
    encoded += layers.Embedding(num_patches, projection_dim)(positions)

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization()(encoded)
        attn = layers.MultiHeadAttention(num_heads, projection_dim)(x1, x1)
        x2 = layers.Add()([attn, encoded])
        x3 = layers.LayerNormalization()(x2)
        mlp = layers.Dense(mlp_dim, activation=tf.nn.gelu)(x3)
        mlp = layers.Dense(projection_dim)(mlp)
        encoded = layers.Add()([mlp, x2])


    # Classification Head

    x = layers.LayerNormalization()(encoded)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return Model(inputs, outputs, name="Hybrid_ResNet50_ViT")

# Build the model
hybrid_model = create_hybrid_resnet_vit()

hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

hybrid_model.summary()


In [ ]:
# Training Hybrid
history_hybrid = hybrid_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=callbacks,  # <-- early stop + checkpoint + LR schedule
    verbose=1
)


In [ ]:
# Test
test_loss_hybrid, test_acc_hybrid = hybrid_model.evaluate(test_ds, verbose=1)
print(f"\n🔥 Hybrid Model Test Accuracy: {test_acc_hybrid:.4f}")
print(f"📉 Test Loss: {test_loss_hybrid:.4f}")


In [ ]:
# Classification Report
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_true = []
y_pred = []
y_proba = []

for images, labels in test_ds:
    preds = hybrid_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))
    y_proba.extend(preds)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_proba = np.array(y_proba)

print("\n🔍 Classification Report — Hybrid ResNet50-ViT")
print(classification_report(y_true, y_pred, digits=4))


In [ ]:
# Confusion Matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix — Hybrid ResNet50-ViT")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()


In [ ]:
# ROC Curves
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

num_classes = 4
y_true_bin = label_binarize(y_true, classes=np.arange(num_classes))

fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

macro_auc = np.mean(list(roc_auc.values()))
print(f"\n🎯 Hybrid Model Macro AUC: {macro_auc:.4f}")


In [ ]:
# ROC Plots
colors = ['blue', 'orange', 'green', 'red']
class_names = ['Class 0', 'Class 1', 'Class 2', 'Class 3']  # Edit if needed

plt.figure(figsize=(8,6))

for i, color in zip(range(num_classes), colors):
    plt.plot(
        fpr[i], tpr[i], lw=2, color=color,
        label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})'
    )

plt.plot([0,1],[0,1],'k--')
plt.xlim([0,1])
plt.ylim([0,1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Hybrid ResNet50-ViT")
plt.legend(loc="lower right")
plt.show()


In [ ]:
# Training vs Validation Curves
acc = history_hybrid.history['accuracy']
val_acc = history_hybrid.history['val_accuracy']
loss = history_hybrid.history['loss']
val_loss = history_hybrid.history['val_loss']

epochs = range(1, len(acc)+1)

plt.figure(figsize=(14,5))

plt.subplot(1,2,1)
plt.plot(epochs, acc, label="Train Acc")
plt.plot(epochs, val_acc, label="Val Acc")
plt.title("Accuracy Curve — Hybrid ResNet50-ViT")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs, loss, label="Train Loss")
plt.plot(epochs, val_loss, label="Val Loss")
plt.title("Loss Curve — Hybrid ResNet50-ViT")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()
